# The volatility smile
What the TICKER1 smile says about the distribution, the no-arbitrage checks on the surface, models that fit the
same smile but price an exotic differently, the Heston parameters on the smile, and the variance swap strike
(Exercises, Equity: Volatility).

In [ ]:
import os
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
import tensorquant as tq

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
vs = eq.vol_surface
S0 = float(eq.spot)
T_NODES, K_NODES = np.asarray(vs.maturity, float), np.asarray(vs.strike, float)
VOLS = vs.volatility_matrix.numpy()


def black_call(F, K, T, vol, df=1.0):
    sd = vol * np.sqrt(T)
    d1 = np.log(F / K) / sd + 0.5 * sd
    return df * (F * norm.cdf(d1) - K * norm.cdf(d1 - sd))

## 1. What the smile says about the distribution
Breeden–Litzenberger: the risk-neutral density of $S_T$ is $\partial^2 C/\partial K^2$ (undiscounted). On the 1Y slice, the
vol is fitted by a cubic in log-moneyness $k = \ln(K/F)$ (smooth, inside the quoted strikes), the call prices are
differentiated twice. Against the lognormal at the ATM vol: the **skew** moves mass to a fat left tail and thins the
right one (negative skewness), the **curvature** fattens both tails (excess kurtosis).

In [ ]:
i = int(np.argmin(np.abs(T_NODES - 1.0)))
T1 = T_NODES[i]
F1 = float(eq.forward(T1))
k_nodes = np.log(K_NODES / F1)
coef = np.polyfit(k_nodes, VOLS[i], 3)
smile = lambda k: np.polyval(coef, k)
k = np.linspace(k_nodes[0], k_nodes[-1], 2001)
K = F1 * np.exp(k)
calls = black_call(F1, K, T1, smile(k))
density = np.gradient(np.gradient(calls, K), K)
atm = smile(0.0)
lognormal = norm.pdf(np.log(K / F1), -0.5 * atm**2 * T1, atm * np.sqrt(T1)) / K


def moments(dens):
    w = dens * np.gradient(K)
    w = w / w.sum()
    x = np.log(K / F1)
    m = (w * x).sum()
    s = np.sqrt((w * (x - m) ** 2).sum())
    return (w * ((x - m) / s) ** 3).sum(), (w * ((x - m) / s) ** 4).sum() - 3


fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(100 * K_NODES / F1, 100 * VOLS[i], "o", label="surface nodes")
ax[0].plot(100 * K / F1, 100 * smile(k), label="cubic fit")
ax[0].axhline(100 * atm, color="grey", ls=":", label="ATM vol")
ax[0].set(xlabel="K / F (%)", ylabel="vol (%)", title=f"1Y smile (T = {T1:.2f})"); ax[0].legend()
ax[1].plot(100 * K / F1, density, label="implied (Breeden-Litzenberger)")
ax[1].plot(100 * K / F1, lognormal, "--", label=f"lognormal at {atm:.1%}")
ax[1].set(xlabel="S_T / F (%)", title="risk-neutral density"); ax[1].legend()
sk, ku = moments(density)
sk0, ku0 = moments(lognormal)
print(f"ln(S_T/F) inside the quoted strikes: skewness {sk:.2f} (lognormal {sk0:.2f}), excess kurtosis {ku:.2f} "
      f"(lognormal {ku0:.2f}); min density {density.min():.2e} (>= 0: no butterfly arbitrage on the fit)")

## 2. No-arbitrage checks on the surface
**Butterfly** (no negative density): call prices convex in the strike, checked on the node strikes of each expiry
(second divided differences $\ge 0$). **Calendar**: total implied variance $w = \sigma^2T$ non-decreasing in $T$ at fixed
forward moneyness (each slice interpolated in $k$). Then the same checks on a surface with the 1Y at-the-money node
pushed down by 10 vol points: its slice is no longer convex, and its total variance falls below the 6M one.

In [ ]:
def checks(vols):
    fwd = np.array([float(eq.forward(t)) for t in T_NODES])
    df = np.array([float(eq.discount_curve.discount(t)) for t in T_NODES])
    butterfly = []
    for a, t in enumerate(T_NODES):
        c = black_call(fwd[a], K_NODES, t, vols[a], df[a])
        slopes = np.diff(c) / np.diff(K_NODES)
        butterfly.append(np.diff(slopes).min())
    k_grid = np.linspace(-0.3, 0.3, 61)
    w = np.array([np.interp(k_grid, np.log(K_NODES / fwd[a]), vols[a] ** 2 * t) for a, t in enumerate(T_NODES)])
    calendar = np.diff(w, axis=0).min(axis=1)
    return pd.DataFrame({"T": T_NODES, "min convexity (butterfly)": butterfly,
                         "min dw to next expiry (calendar)": list(calendar) + [np.nan]}).set_index("T")


print("market surface"); display(checks(VOLS).round(6))
bad = VOLS.copy()
bad[3, 4] -= 0.10                                                  # 1Y at the money, 10 vol points lower
print("1Y ATM node 10 vol points lower"); checks(bad).round(6)

## 3. Same smile, different exotic prices
Local vol and Heston SLV with two mixings, all calibrated to the TICKER1 surface: the 1Y at-the-money call is the
same in every model (all reprice the vanillas), the barriers are not. A vanilla depends on the distribution of $S_T$
only, a barrier on the whole path, i.e. on the dynamics of the smile (the forward smile), which the vanillas do not
pin down: the mixing is a model choice, and the spread of the exotic prices across it is model risk.

In [ ]:
import time
EXPIRY = mkt.evaluation_date + timedelta(days=365)
products = {
    "ATM call": lambda: tq.VanillaOption(eq.currency, mkt.evaluation_date, EXPIRY, tq.OptionType.Call, S0,
                                         underlying="TICKER1"),
    "up-and-out call 100 / 120": lambda: tq.BarrierOption(eq.currency, mkt.evaluation_date, EXPIRY, tq.OptionType.Call,
                                                          S0, barrier=1.2 * S0, barrier_type=tq.BarrierType.UpOut,
                                                          underlying="TICKER1"),
    "down-and-out put 100 / 80": lambda: tq.BarrierOption(eq.currency, mkt.evaluation_date, EXPIRY, tq.OptionType.Put,
                                                          S0, barrier=0.8 * S0, barrier_type=tq.BarrierType.DownOut,
                                                          underlying="TICKER1"),
}
local_vol = tq.LocalVolatilityModel.calibrate(eq)
heston = tq.HestonModel.calibrate(eq)
models = {"local vol": local_vol,
          "SLV mixing 0.5": tq.HestonSLVModel(heston, local_vol, mixing=0.5).calibrate_leverage(eq),
          "SLV mixing 1.0": tq.HestonSLVModel(heston, local_vol, mixing=1.0).calibrate_leverage(eq)}
table = {}
for name, model in models.items():
    pricer = tq.VanillaMCPricer(model=model, n_paths=50_000, n_steps=100, seed=4)
    row = {}
    for label, make in products.items():
        o = make()
        pricer.price(o, mkt)
        row[label] = f"{float(o.price):.3f} ({float(o.standard_error):.3f})"
    table[name] = row
pd.DataFrame(table).T                                             # price (Monte Carlo standard error)

## 4. Heston parameters and the smile
Closed-form Heston smiles (COS) on the TICKER1 forward, one parameter moved at a time around a base set: $v_0$ moves
the short-dated level, $\theta$ the long-dated level, $\kappa$ how fast one goes to the other, $\xi$ (vol of vol) the
curvature, $\rho$ the skew. The skew decays with the expiry; one set of constant parameters cannot follow both the
steep short end and the flat back of the market's skew term structure (below: under the market at 1M, above it from
3M on). A steep short-dated skew needs a large $\xi$, which then over-curves the long expiries.

In [ ]:
BASE = dict(v0=0.05, kappa=1.5, theta=0.06, xi=0.6, rho=-0.7)
moneyness = np.linspace(0.6, 1.4, 33)


def smile_1y(**params):
    proc = tq.HestonModel(**{**BASE, **params}).process(eq)
    f = float(eq.forward(1.0))
    return [100 * proc.implied_vol(m * f, 1.0) for m in moneyness]


bumps = {"rho": (-0.9, -0.3), "xi": (0.3, 1.0), "v0": (0.03, 0.08), "kappa": (0.5, 4.0)}
fig, axes = plt.subplots(1, 4, figsize=(20, 3.8), sharey=True)
for ax, (name, values) in zip(axes, bumps.items()):
    ax.plot(100 * moneyness, smile_1y(), "k", label=f"base {name}={BASE[name]}")
    for v in values:
        ax.plot(100 * moneyness, smile_1y(**{name: v}), label=f"{name}={v}")
    ax.set(title=f"1Y smile vs {name}", xlabel="K / F (%)"); ax.legend(fontsize=8)
axes[0].set_ylabel("implied vol (%)")
plt.tight_layout()

In [ ]:
# skew (vol at 90% minus 110% of the forward) by expiry: Heston (base) vs the TICKER1 market
proc = tq.HestonModel(**BASE).process(eq)
rows = []
for t in T_NODES:
    f = float(eq.forward(t))
    rows.append({"T": t, "Heston skew 90-110 (vol pts)": 100 * (proc.implied_vol(0.9 * f, t) - proc.implied_vol(1.1 * f, t)),
                 "market skew 90-110 (vol pts)": 100 * (float(vs.volatility(strike=0.9 * f, tenor=t))
                                                        - float(vs.volatility(strike=1.1 * f, tenor=t)))})
pd.DataFrame(rows).set_index("T").round(2)

## 5. The variance swap strike
A variance swap pays $\sigma^2_{\text{realised}} - K_{\text{var}}$. Its fair strike is replicated by a static strip of
out-of-the-money options weighted $1/K^2$ (plus a delta hedge):
$$K_{\text{var}} = \frac{2}{T\,P(0,T)}\Big(\int_0^F \frac{P(K)}{K^2}\,dK + \int_F^\infty \frac{C(K)}{K^2}\,dK\Big).$$
With a flat smile it gives the flat vol back; with a skew the cheap-strike puts weigh most, so $\sqrt{K_{\text{var}}}$ is
above the ATM vol, and how far depends on how the wings are extrapolated (here flat beyond the quoted strikes).

In [ ]:
def var_strike(t, vol_fn, lo=0.05, hi=4.0, n=4000):
    f = float(eq.forward(t))
    strikes = f * np.linspace(lo, hi, n)
    vols = np.array([vol_fn(k, t) for k in strikes])
    calls = black_call(f, strikes, t, vols)                         # undiscounted, = integrand / P(0,T)
    puts = calls - (f - strikes)
    otm = np.where(strikes < f, puts, calls)
    return 2 / t * np.trapezoid(otm / strikes**2, strikes)


market_vol = lambda k, t: float(vs.volatility(strike=k, tenor=t))
rows = []
for t in (0.25, 1.0, 3.0):
    f = float(eq.forward(t))
    atm_vol = market_vol(f, t)
    rows.append({"T": t, "ATM vol (%)": 100 * atm_vol,
                 "var swap vol (%)": 100 * np.sqrt(var_strike(t, market_vol)),
                 "flat smile at ATM (%)": 100 * np.sqrt(var_strike(t, lambda k, tt: atm_vol)),
                 "strikes 70-130% only (%)": 100 * np.sqrt(var_strike(t, market_vol, 0.7, 1.3))})
pd.DataFrame(rows).set_index("T").round(2)

## Exercises
1. Steepen the 1Y skew (vol at 60% up 5 points, at 140% down 2): how do the density, the skewness and the variance
   swap strike move?
2. Price the down-and-in put with Heston SLV at mixing 0.1 and 1.0: which model does each end resemble?
3. Find a Heston parameter set whose 3M skew matches the market: what vol of vol does it need, and what does it do to
   the 5Y smile?